# 💳 Credit Card Fraud Detection
## Step 2 — Train Models (XGBoost | Random Forest | Decision Tree)

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import time

# Load preprocessed data from Step 1
X_train, X_test, y_train, y_test = joblib.load('data/train_test_split.pkl')
print(f'Training samples: {X_train.shape[0]:,}')
print(f'Test samples:     {X_test.shape[0]:,}')

### Model 1 — Decision Tree

In [ ]:
print('Training Decision Tree...')
t0 = time.time()

dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42
)
dt_model.fit(X_train, y_train)

dt_time = time.time() - t0
dt_preds = dt_model.predict(X_test)
dt_proba = dt_model.predict_proba(X_test)[:, 1]

print(f'Done in {dt_time:.1f}s')
print(f'ROC-AUC: {roc_auc_score(y_test, dt_proba):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, dt_preds, target_names=['Legitimate', 'Fraud']))

joblib.dump(dt_model, 'models/decision_tree.pkl')
print('Model saved → models/decision_tree.pkl')

### Model 2 — Random Forest

In [ ]:
print('Training Random Forest (this may take a few minutes)...')
t0 = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    class_weight='balanced',
    n_jobs=-1,          # use all CPU cores
    random_state=42
)
rf_model.fit(X_train, y_train)

rf_time = time.time() - t0
rf_preds = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

print(f'Done in {rf_time:.1f}s')
print(f'ROC-AUC: {roc_auc_score(y_test, rf_proba):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, rf_preds, target_names=['Legitimate', 'Fraud']))

joblib.dump(rf_model, 'models/random_forest.pkl')
print('Model saved → models/random_forest.pkl')

### Model 3 — XGBoost

In [ ]:
print('Training XGBoost...')
t0 = time.time()

# scale_pos_weight handles class imbalance in XGBoost
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1,
    random_state=42
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

xgb_time = time.time() - t0
xgb_preds = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print(f'Done in {xgb_time:.1f}s')
print(f'ROC-AUC: {roc_auc_score(y_test, xgb_proba):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, xgb_preds, target_names=['Legitimate', 'Fraud']))

joblib.dump(xgb_model, 'models/xgboost.pkl')
print('Model saved → models/xgboost.pkl')

### Quick Summary

In [ ]:
summary = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'XGBoost'],
    'ROC-AUC': [
        round(roc_auc_score(y_test, dt_proba), 4),
        round(roc_auc_score(y_test, rf_proba), 4),
        round(roc_auc_score(y_test, xgb_proba), 4)
    ],
    'Train Time (s)': [round(dt_time,1), round(rf_time,1), round(xgb_time,1)]
})
print(summary.to_string(index=False))
print('\n✅ All models trained and saved to models/')